In [0]:
from pyspark.sql.functions import current_timestamp, lit, col
import re

# Source
catalog_name = "my_catalog"
source_schema = "landing"
volume_name = "raw_files"

# Target
target_schema = "bronze"

source_path = f"/Volumes/{catalog_name}/{source_schema}/{volume_name}/"

# Get all files
files = dbutils.fs.ls(source_path)

# Process CSV files
for file in files:

    if file.name.lower().endswith(".csv"):

        print(f"Processing {file.name}")

        # Generate table name from file name
        table_name = (
            re.sub(r"[^a-zA-Z0-9_]", "_",
                   file.name.replace(".csv", "").lower())
        )

        full_table_name = (
            f"{catalog_name}.{target_schema}.{table_name}"
        )

        # Read CSV
        df = (
            spark.read
            .format("csv")
            .option("header", "true")
            .option("inferSchema", "true")
            .load(file.path)
        )

        # Audit columns
        df = (
            df
            .withColumn("_source_file", col("_metadata.file_path"))
            .withColumn("_load_timestamp", current_timestamp())
            .withColumn("_bronze_layer", lit("bronze"))
        )

        # Write Delta table
        (
            df.write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .saveAsTable(full_table_name)
        )

        print(f"Created table: {full_table_name}")

print("Bronze ingestion complete")